In [1]:
%load_ext autoreload
%autoreload 2

import fastf1
from src.cleaning import clean_laps, add_gap_ahead

fastf1.Cache.configure(cache_dir='cache')
session = fastf1.get_session(2023, 'Bahrain', 'R')
session.load(telemetry = False, weather = False, messages = False)
laps = session.laps



core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.9.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


In [2]:
cleaned, funnel = clean_laps(laps)
print('Laps at the start:', len(laps), 'Laps left:', len(cleaned))
funnel

Laps at the start: 1056 Laps left: 889


,Step,Laps left,Laps removed
0,Start,1056,0
1,Remove in laps and out laps,954,102
2,Remove lap 1,934,20
3,Remove inaccurate laps,914,20
4,Remove laps with no lap time,914,0
5,Keep only green flag laps,889,25
6,Remove laps after a Safety Car lap,889,0


In [3]:
print(cleaned[['Driver', 'LapNumber', 'LapTime', 'LapTimeSeconds']].head())
print('Fastest lap:', cleaned['LapTimeSeconds'].min())

  Driver  LapNumber                LapTime  LapTimeSeconds
2    VER        3.0 0 days 00:01:38.006000          98.006
3    VER        4.0 0 days 00:01:37.976000          97.976
4    VER        5.0 0 days 00:01:38.035000          98.035
5    VER        6.0 0 days 00:01:37.986000          97.986
6    VER        7.0 0 days 00:01:38.021000          98.021
Fastest lap: 93.996


In [4]:
with_gaps = add_gap_ahead(laps, cleaned)
print('Laps:', len(with_gaps))
print('Laps with a gap:', with_gaps['GapAhead'].notna().sum(), 'No car ahead:', with_gaps['GapAhead'].isna().sum())
print('Laps in traffic:', with_gaps['InTraffic'].sum())

Laps: 889
Laps with a gap: 840 No car ahead: 49
Laps in traffic: 134


In [5]:
ver = with_gaps[with_gaps['Driver'] == 'VER']
print('Verstappen laps left:', len(ver))
print(ver.groupby('Compound')['LapTimeSeconds'].agg(['count', 'min', 'max']))
print('Missing laps:', sorted(set(range(1, int(laps['LapNumber'].max()) + 1)) - set(ver['LapNumber'])))

Verstappen laps left: 49
          count     min     max
Compound                       
HARD         18  96.236  97.427
SOFT         31  97.372  98.591
Missing laps: [1, 2, 14, 15, 36, 37, 41, 42]


In [6]:
with_gaps[(with_gaps['Driver'] == 'SAI') & (with_gaps['LapNumber'].between(43, 47))][['LapNumber', 'LapTimeSeconds', 'DriverAhead', 'GapAhead', 'InTraffic']]

,LapNumber,LapTimeSeconds,DriverAhead,GapAhead,InTraffic
766,43.0,97.294,PER,23.729,False
767,44.0,97.389,PER,24.140,False
768,45.0,99.561,ALO,0.988,True
769,46.0,97.844,ALO,1.745,False
770,47.0,98.222,ALO,2.641,False


## Testing on the Saudi Arabian Grand Prix

In [7]:
saudi = fastf1.get_session(2023, 'Saudi Arabia', 'R')
saudi.load(telemetry = False, weather = False, messages = False)
saudi_laps = saudi.laps 

saudi_cleaned, saudi_funnel = clean_laps(saudi_laps, remove_restarts = False)
saudi_gaps = add_gap_ahead(saudi_laps, saudi_cleaned)

print('Laps:', len(saudi_gaps))
print('Laps with a gap:', saudi_gaps['GapAhead'].notna().sum(), 'No car ahead:', saudi_gaps['GapAhead'].isna().sum())
print('Laps in traffic:', saudi_gaps['InTraffic'].sum())
saudi_funnel

core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.9.0]
req            INFO 	Using cached data for session_info


req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 11 completed the race distance 00:00.026000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['11', '1', '14', '63', '44', '55', '16', '31', '10', '20', '22', '27', '24', '21', '81', '2', '4', '77', '23', '18']


Laps: 824
Laps with a gap: 779 No car ahead: 45
Laps in traffic: 206


,Step,Laps left,Laps removed
0,Start,943,0
1,Remove in laps and out laps,894,49
2,Remove lap 1,875,19
3,Remove inaccurate laps,829,46
4,Remove laps with no lap time,829,0
5,Keep only green flag laps,824,5
6,Remove laps after a Safety Car lap,824,0


In [8]:
sc_laps = saudi_laps[saudi_laps['TrackStatus'].str.contains('4')]
print('Safety Car laps at the start:', len(sc_laps))
print('Safety Car laps left:',
      saudi_gaps['TrackStatus'].str.contains('4').sum())
print(sc_laps['LapNumber'].value_counts().sort_index())

print('Removed as in or out laps:', len(sc_laps) - len(sc_laps.pick_wo_box()))
print('Marked inaccurate:', (sc_laps.pick_wo_box()['IsAccurate'] == False).sum())

Safety Car laps at the start: 72
Safety Car laps left: 0
LapNumber
17.0    15
18.0    19
19.0    19
20.0    19
Name: count, dtype: int64
Removed as in or out laps: 26
Marked inaccurate: 46


In [9]:
prev_status = saudi_laps.groupby('Driver')['TrackStatus'].shift(1)
after_sc = saudi_laps[prev_status.str.contains('4', na = False)]
print('Laps straight after a Safety Car lap:', len(after_sc))
print(after_sc['LapNumber'].value_counts().sort_index())

left = saudi_gaps.merge(after_sc[['Driver', 'LapNumber']], on = ['Driver', 'LapNumber'])
print('Left after cleaning:', len(left))
left[['Driver', 'LapNumber', 'TrackStatus', 'LapTimeSeconds']]

Laps straight after a Safety Car lap: 71
LapNumber
18.0    14
19.0    19
20.0    19
21.0    19
Name: count, dtype: int64
Left after cleaning: 19


,Driver,LapNumber,TrackStatus,LapTimeSeconds
0,VER,21.0,1,96.946
1,GAS,21.0,1,98.374
2,PER,21.0,1,94.557
3,ALO,21.0,1,96.033
4,LEC,21.0,1,98.031
5,SAR,21.0,1,98.699
6,MAG,21.0,1,98.649
7,DEV,21.0,1,99.388
8,TSU,21.0,1,98.152
9,ALB,21.0,1,99.011


In [10]:
saudi_gaps['SlowerThanStintMedian'] = saudi_gaps['LapTimeSeconds'] - saudi_gaps.groupby(['Driver', 'Stint'])['LapTimeSeconds'].transform('median')
restart = saudi_gaps['LapNumber'] == 21
print(saudi_gaps.groupby(restart)['SlowerThanStintMedian'].agg(['count', 'mean', 'median', 'min', 'max']).round(3))

slowest = saudi_gaps.sort_values('SlowerThanStintMedian',
                                 ascending = False).head(19)
print('Restart laps among the 19 slowest laps:', (slowest['LapNumber'] == 21).sum())

           count   mean  median    min    max
LapNumber                                    
False        805  0.059  -0.001 -1.765  3.901
True          19  3.421   3.508  1.779  4.169
Restart laps among the 19 slowest laps: 17


### Re-testing with the restart-lap rule

In [11]:
saudi_fixed, saudi_fixed_funnel = clean_laps(saudi_laps)
print('Restart laps left:', saudi_fixed.index.isin(after_sc.index).sum())
saudi_fixed_funnel

Restart laps left: 0


,Step,Laps left,Laps removed
0,Start,943,0
1,Remove in laps and out laps,894,49
2,Remove lap 1,875,19
3,Remove inaccurate laps,829,46
4,Remove laps with no lap time,829,0
5,Keep only green flag laps,824,5
6,Remove laps after a Safety Car lap,805,19
